# BYOS: Sustainability Reports + Judge+Critic Evaluation (H3)

This notebook covers the final two stages of the pipeline:
1. **Report Generation** — aggregate per-user emissions stats and call Claude to write a sustainability report
2. **Judge+Critic Agent** — evaluate each report's factual accuracy using a two-pass Claude loop
3. **H3 Evaluation** — compare Judge+Critic vs. single-prompt judge using Cohen's κ against human labels

**Prerequisites:** run `modeling.ipynb` first — this notebook loads `models/lgbm_mode_classifier.pkl`
and reconstructs `em_df` from the saved model and feature cache.

**Requires:** `ANTHROPIC_API_KEY` set in `.env`

## 0. Setup

This cell does the following:

1. **Imports & environment** — loads `.env` to read `ANTHROPIC_API_KEY`
2. **Initialises Claude client** — `claude-sonnet-4-6` used for both report generation and Judge+Critic
3. **Loads the pickled model** — reads `models/lgbm_mode_classifier.pkl` saved by `modeling.ipynb` (includes the LightGBM model, feature column list, and label encoder)
4. **Reconstructs the test split** — re-runs the same `StratifiedGroupKFold(n_splits=5, random_state=42)` as `modeling.ipynb` so we get the identical held-out users
5. **Builds `em_df`** — runs the model on the test split and attaches predicted mode + CO₂ per window (overlap-corrected: each window's distance × 0.5 to avoid double-counting from 50% stride)

In [1]:
import sys
import os
import json
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
from pathlib import Path
from dotenv import load_dotenv
import anthropic

warnings.filterwarnings('ignore')
plt.rcParams['figure.dpi'] = 120
sys.path.insert(0, str(Path.cwd()))
load_dotenv()

# ── Configure Claude ─────────────────────────────────────────────────────────
claude = anthropic.Anthropic(api_key=os.environ['ANTHROPIC_API_KEY'])
CLAUDE_MODEL = 'claude-sonnet-4-6'

# ── Load saved model artifact ─────────────────────────────────────────────────
MODEL_PATH = Path('../models/lgbm_mode_classifier.pkl')
with open(MODEL_PATH, 'rb') as f:
    artifact = pickle.load(f)

model        = artifact['model']
FEATURE_COLS = artifact['feature_cols']
le           = artifact['label_encoder']
print(f'Model loaded — macro-F1: {artifact["macro_f1"]:.3f}')
print(f'Features: {FEATURE_COLS}')
print(f'Classes:  {list(le.classes_)}')

# ── Reconstruct test split (same seed as modeling.ipynb) ─────────────────────
from sklearn.model_selection import StratifiedGroupKFold
from features import build_feature_dataset
import kagglehub

path     = kagglehub.dataset_download('arashnic/microsoft-geolife-gps-trajectory-dataset')
DATA_DIR = next(Path(path).rglob('Data'))
feature_df = build_feature_dataset(DATA_DIR)
feature_df['mode'] = pd.Categorical(feature_df['mode'], categories=sorted({'walk','bike','bus','car'}))

df = feature_df.dropna(subset=FEATURE_COLS).copy()
sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)
train_idx, test_idx = next(sgkf.split(df, y=df['mode'], groups=df['user']))
test_df = df.iloc[test_idx].copy()

# ── Build em_df with predictions ─────────────────────────────────────────────
EMISSION_FACTORS = {'car': 170, 'bus': 89, 'bike': 0, 'walk': 0}
y_pred = model.predict(test_df[FEATURE_COLS])
em_df  = test_df.copy()
em_df['pred_mode']   = le.inverse_transform(y_pred)
em_df['dist_km']     = em_df['distance_total_m'] / 1000
em_df['dist_km_adj'] = em_df['dist_km'] * 0.5
em_df['co2_g']       = em_df['pred_mode'].map(EMISSION_FACTORS).fillna(0) * em_df['dist_km_adj']

print(f'\nTest users: {sorted(em_df["user"].unique())}')
print(f'Windows:    {len(em_df):,}')


Model loaded — macro-F1: 0.565
Features: ['speed_mean', 'speed_max', 'speed_std', 'accel_mean', 'accel_std', 'jerk_mean', 'stop_ratio', 'bearing_variance', 'distance_total_m']
Classes:  ['bike', 'bus', 'car', 'walk']
Loading features from cache: /home/mnguyen0226/Documents/personal/repositories/geolife-human-psychology/data/processed/features.parquet
Loaded 614,460 windows

Test users: ['052', '068', '110', '112', '129', '179']
Windows:    143,023


---
## 1. Sustainability Report Generation

For each test user we:
1. Aggregate emissions stats from `em_df` into a structured dict
2. Call GPT-4.1 with the stats and a factuality constraint
3. Save the report alongside the raw stats for Judge+Critic evaluation

**Factuality constraint:** every number in the report must come directly from the data.
This sets up verifiable claims for H3.

In [2]:
def build_user_stats(user: str, em_df) -> dict:
    """Aggregate emissions stats for one user."""
    u        = em_df[em_df['user'] == user]
    mode_km  = u.groupby('pred_mode')['dist_km_adj'].sum().round(2).to_dict()
    mode_co2 = u.groupby('pred_mode')['co2_g'].sum().div(1000).round(2).to_dict()
    total_co2 = round(sum(mode_co2.values()), 2)
    car_co2   = mode_co2.get('car', 0)
    return {
        'user':           user,
        'mode_km':        mode_km,
        'mode_co2_kg':    mode_co2,
        'total_co2_kg':   total_co2,
        'car_pct_of_co2': round(100 * car_co2 / total_co2, 1) if total_co2 else 0,
    }


def generate_report(stats: dict) -> str:
    """Call Claude to generate a sustainability report from user stats."""
    prompt = (
        'You are a sustainability analyst writing a personal mobility report.\n'
        'Rules:\n'
        '  1. Write exactly 4-6 sentences.\n'
        '  2. Every number you state must come directly from the data below — '
        'do not round, estimate, or invent figures.\n'
        '  3. End with one concrete, actionable recommendation based on the data.\n\n'
        f'User mobility data:\n{json.dumps(stats, indent=2)}\n\n'
        'Report:'
    )
    return claude.messages.create(
        model=CLAUDE_MODEL,
        max_tokens=350,
        messages=[{'role': 'user', 'content': prompt}],
    ).content[0].text.strip()


# ── User selection: keep only users with ≥2 distinct predicted modes ─────────
all_test_users = sorted(em_df['user'].unique())
print(f'All test users ({len(all_test_users)}): {all_test_users}')

user_mode_counts = (
    em_df.groupby('user')['pred_mode']
    .nunique()
    .rename('n_modes')
)
print('\nDistinct predicted modes per user:')
print(user_mode_counts.to_string())

dropped = user_mode_counts[user_mode_counts < 2].index.tolist()
selected_users = user_mode_counts[user_mode_counts >= 2].index.tolist()

if dropped:
    print(f'\nDropped (single-mode, boring reports): {dropped}')
print(f'Selected for report generation ({len(selected_users)}): {selected_users}')

# ── Generate reports ──────────────────────────────────────────────────────────
reports = {}
for user in selected_users:
    stats  = build_user_stats(user, em_df)
    report = generate_report(stats)
    reports[user] = {'stats': stats, 'report': report}
    print(f'\n--- User {user} ---')
    print(f'Total CO₂: {stats["total_co2_kg"]} kg | Car: {stats["car_pct_of_co2"]}%')
    print(report)

print(f'\n{len(reports)} reports generated.')


All test users (6): ['052', '068', '110', '112', '129', '179']

Distinct predicted modes per user:
user
052    4
068    4
110    3
112    4
129    4
179    4
Selected for report generation (6): ['052', '068', '110', '112', '129', '179']

--- User 052 ---
Total CO₂: 167.86 kg | Car: 64.4%
User 052 traveled a combined total across all modes, with bus accounting for 670.52 km and car accounting for 636.33 km as the two dominant modes, while walking contributed 214.35 km and cycling 58.58 km at zero emissions. Despite the car being traveled slightly less distance than the bus, it generated 108.18 kg of CO₂ compared to only 59.68 kg from the bus, making it responsible for 64.4% of the user's total 167.86 kg CO₂ footprint. This disproportionate carbon burden from car travel highlights a clear opportunity for emissions reduction, particularly given that the user already demonstrates a willingness to use lower-carbon modes. **Recommendation:** Shift at least a portion of the 636.33 km currentl

In [3]:
REPORTS_DIR = Path('../reports')
REPORTS_DIR.mkdir(exist_ok=True)

out_path = REPORTS_DIR / 'user_reports.json'
with open(out_path, 'w') as f:
    json.dump(reports, f, indent=2)

print(f'Saved {len(reports)} reports → {out_path}')
print('Next: section 2 — Judge+Critic evaluation')


Saved 6 reports → ../reports/user_reports.json
Next: section 2 — Judge+Critic evaluation


---
## 2. Judge+Critic Agent (H3)

**Goal:** Evaluate each report's factual accuracy using a two-pass LLM loop, then
compare against a single-prompt baseline to test whether the loop improves agreement
with human labels (Cohen's κ).

**Loop:**
```
Report + raw stats
    ↓  Judge pass — verify each numeric claim
Judge verdicts
    ↓  Critic pass — flag weak justifications
Critic flags
    ↓  Judge revises flagged verdicts
Final verdicts: {correct, incorrect, unverifiable} per claim
```

**Baseline:** single Claude prompt with same inputs, no Critic pass.

In [4]:
def _call(prompt: str, max_tokens: int = 600) -> str:
    return claude.messages.create(
        model=CLAUDE_MODEL,
        max_tokens=max_tokens,
        messages=[{'role': 'user', 'content': prompt}],
    ).content[0].text.strip()


def judge_report(stats: dict, report: str) -> str:
    """Judge pass: verify each numeric claim in the report."""
    prompt = (
        'You are a fact-checker for sustainability reports.\n'
        'Given the raw data and the report, identify each numeric claim in the report\n'
        'and label it: correct / incorrect / unverifiable.\n'
        'Cite the specific data value that supports or contradicts each claim.\n\n'
        f'Raw data:\n{json.dumps(stats, indent=2)}\n\n'
        f'Report:\n{report}\n\n'
        'List each claim and verdict:'
    )
    return _call(prompt, max_tokens=600)


def critic_pass(judge_output: str) -> str:
    """Critic pass: identify weak justifications in the Judge's assessment."""
    prompt = (
        'You are reviewing a fact-checker\'s assessment of a sustainability report.\n'
        'Identify any claims the fact-checker accepted too easily, any verdicts\n'
        'that lack clear data support, or any missed claims.\n'
        'Be specific about which verdicts need revision and why.\n\n'
        f'Fact-checker assessment:\n{judge_output}\n\n'
        'Flags for revision:'
    )
    return _call(prompt, max_tokens=400)


def judge_revised(stats: dict, report: str, critic_flags: str) -> str:
    """Judge revises verdicts based on Critic flags."""
    prompt = (
        'You previously assessed a sustainability report. A critic has flagged issues\n'
        'with your assessment. Revise your verdicts where the critic raises valid points.\n\n'
        f'Raw data:\n{json.dumps(stats, indent=2)}\n\n'
        f'Report:\n{report}\n\n'
        f'Critic flags:\n{critic_flags}\n\n'
        'Final revised verdicts (correct / incorrect / unverifiable per claim):'
    )
    return _call(prompt, max_tokens=600)


def baseline_judge(stats: dict, report: str) -> str:
    """Single-prompt baseline — no Critic pass."""
    prompt = (
        'Given the data and the report, is each numeric claim correct, incorrect,\n'
        'or unverifiable? List each claim with its verdict.\n\n'
        f'Data:\n{json.dumps(stats, indent=2)}\n\n'
        f'Report:\n{report}\n\n'
        'Verdicts:'
    )
    return _call(prompt, max_tokens=600)


# ── Run Judge+Critic loop on all reports ──────────────────────────────────────
eval_results = {}
for user, data in reports.items():
    stats  = data['stats']
    report = data['report']

    j1       = judge_report(stats, report)
    critic   = critic_pass(j1)
    j_final  = judge_revised(stats, report, critic)
    baseline = baseline_judge(stats, report)

    eval_results[user] = {
        'stats':          stats,
        'report':         report,
        'judge_initial':  j1,
        'critic':         critic,
        'judge_final':    j_final,
        'baseline_judge': baseline,
    }
    print(f'\n=== User {user} ===')
    print(f'--- Baseline judge ---\n{baseline[:300]}')
    print(f'--- Final (loop) judge ---\n{j_final[:300]}')

# Save for H3 Cohen's κ computation
out_path = Path('../reports/eval_results.json')
with open(out_path, 'w') as f:
    json.dump(eval_results, f, indent=2)
print(f'\nSaved eval results → {out_path}')
print('Next: hand-label claims → compute Cohen\'s κ (section 3)')



=== User 052 ===
--- Baseline judge ---
I'll check each numeric claim in the report against the data.

---

**Claim 1: Bus accounts for 670.52 km**
✅ **Correct** — matches `mode_km.bus: 670.52`

**Claim 2: Car accounts for 636.33 km**
✅ **Correct** — matches `mode_km.car: 636.33`

**Claim 3: Walking contributed 214.35 km**
✅ **Correct** —
--- Final (loop) judge ---
# Revised Fact-Check Verdicts — User 052 Sustainability Report

---

## Claims 1–7 and 10: ✅ CORRECT
These are straightforward data lookups or simple comparisons accurately drawn from the raw data. Verdicts stand unchanged.

---

## Claim 8: Car responsible for 64.4% of total CO₂ — ✅ CORRECT (verdic

=== User 068 ===
--- Baseline judge ---
I'll check each numeric claim in the report against the data.

---

**1. "combined total across all modes of 4,720.12 km"**
1449.54 + 1909.57 + 893.3 + 467.71 = **4,720.12 km** ✅ **Correct**

---

**2. "bus … 1,909.57 km"**
Matches data exactly. ✅ **Correct**

---

**3. "bike at 1,449.54 

In [7]:
# ── Generate human_labels.csv template from eval_results ─────────────────────
# Workflow:
#   1. Run this cell — creates CSV with baseline_label and loop_label auto-filled
#   2. Open reports/human_labels.csv in a spreadsheet
#   3. For each row, read the claim and check it against source_data
#      → fill ONLY human_label: correct / incorrect / unverifiable
#   4. Save CSV and run Section 3 to compute Cohen's κ

import re

def extract_claims(report: str) -> list:
    """Extract sentences containing numeric values as candidate claims."""
    sentences = re.split(r'(?<=[.!?])\s+', report.strip())
    return [s for s in sentences if re.search(r'\d', s)]


def parse_verdict(llm_output: str, claim: str) -> str:
    """Find the verdict for a claim in an LLM judge output.
    Looks for 'correct', 'incorrect', or 'unverifiable' near the claim text.
    Returns the verdict string or 'unverifiable' if not found.
    """
    # Search within a window around the claim's key numbers
    numbers = re.findall(r'\d+\.?\d*', claim)
    output_lower = llm_output.lower()
    # Find the region of the output most relevant to this claim
    best_pos = -1
    for num in numbers:
        pos = output_lower.find(num)
        if pos != -1 and (best_pos == -1 or pos < best_pos):
            best_pos = pos
    if best_pos == -1:
        return 'unverifiable'
    # Look for a verdict label within ±300 chars of the number
    window = output_lower[max(0, best_pos - 100): best_pos + 300]
    if 'incorrect' in window:
        return 'incorrect'
    elif 'correct' in window:
        return 'correct'
    elif 'unverifiable' in window or 'cannot' in window or 'not verifiable' in window:
        return 'unverifiable'
    return 'unverifiable'


rows = []
for user, data in eval_results.items():
    claims   = extract_claims(data['report'])
    stats    = data['stats']
    baseline = data['baseline_judge']
    loop     = data['judge_final']
    full_source = (
        f"mode_km={json.dumps(stats['mode_km'])} | "
        f"mode_co2_kg={json.dumps(stats['mode_co2_kg'])} | "
        f"total_co2_kg={stats['total_co2_kg']} | "
        f"car_pct_of_co2={stats['car_pct_of_co2']}%"
    )
    for claim in claims:
        rows.append({
            'user':           user,
            'claim':          claim,
            'source_data':    full_source,
            'human_label':    '',
            'baseline_label': parse_verdict(baseline, claim),
            'loop_label':     parse_verdict(loop, claim),
        })

labels_df = pd.DataFrame(rows)
labels_path = Path('../reports/human_labels.csv')
labels_df.to_csv(labels_path, index=False)

print(f'Generated {len(labels_df)} claims across {labels_df["user"].nunique()} users')
print(f'Saved template → {labels_path}')
print('\nOnly fill in the human_label column (correct / incorrect / unverifiable).')
print('baseline_label and loop_label are auto-filled from the LLM outputs.')
print('\nPreview:')
for _, row in labels_df.iterrows():
    print(f'  [{row["user"]}] {row["claim"][:80]}')
    print(f'    baseline={row["baseline_label"]}  loop={row["loop_label"]}')
    print()


Generated 28 claims across 6 users
Saved template → ../reports/human_labels.csv

Only fill in the human_label column (correct / incorrect / unverifiable).
baseline_label and loop_label are auto-filled from the LLM outputs.

Preview:
  [052] User 052 traveled a combined total across all modes, with bus accounting for 670
    baseline=correct  loop=correct

  [052] Despite the car being traveled slightly less distance than the bus, it generated
    baseline=correct  loop=correct

  [052] **Recommendation:** Shift at least a portion of the 636.33 km currently driven b
    baseline=correct  loop=unverifiable

  [068] User 068 traveled a combined total across all modes of 4,720.12 km, with bus rep
    baseline=correct  loop=correct

  [068] Despite the car accounting for a relatively smaller distance traveled, it contri
    baseline=correct  loop=correct

  [068] The bus segment, while the longest in distance, generated 169.95 kg of CO₂, maki
    baseline=correct  loop=correct

  [068] Nota

---
## 3. H3 Evaluation — Cohen's κ

**Goal:** Quantify whether the Judge+Critic loop agrees with human labels better
than the single-prompt baseline.

**Steps:**
1. Extract individual numeric claims from each report
2. Hand-label each claim: `correct` / `incorrect` / `unverifiable`
3. Parse loop and baseline verdicts into the same labels
4. Compute Cohen's κ for each

Save human labels to `reports/human_labels.csv` with columns:
`user, claim, source_value, human_label`

In [8]:
from sklearn.metrics import cohen_kappa_score

# Load human labels — fill this CSV after hand-labeling
# Format: user, claim, source_value, human_label, baseline_label, loop_label
labels_path = Path('../reports/human_labels.csv')

if labels_path.exists():
    labels_df = pd.read_csv(labels_path)
    human    = labels_df['human_label'].tolist()
    baseline = labels_df['baseline_label'].tolist()
    loop     = labels_df['loop_label'].tolist()

    k_baseline = cohen_kappa_score(human, baseline)
    k_loop     = cohen_kappa_score(human, loop)

    print(f'Claims evaluated : {len(human)}')
    print(f'Baseline κ       : {k_baseline:.3f}')
    print(f'Loop κ           : {k_loop:.3f}')
    print(f'Delta            : {k_loop - k_baseline:+.3f}')
    print()
    if k_loop > k_baseline:
        print('H3 supported — Judge+Critic loop agrees with humans more than baseline')
    else:
        print('H3 not supported — loop does not improve over baseline')
    print('Report both numbers regardless of direction.')
else:
    print(f'human_labels.csv not found at {labels_path}')
    print('Complete hand-labeling first, then re-run this cell.')


Claims evaluated : 28
Baseline κ       : 0.000
Loop κ           : 0.232
Delta            : +0.232

H3 supported — Judge+Critic loop agrees with humans more than baseline
Report both numbers regardless of direction.
